# Electronic Online Shop - Machine Learning Preprocessing

This notebook prepares the cleaned Electronic Online Shop dataset for Machine Learning.

Included steps:
- Dataset loading and overview
- Feature/target separation
- Label Encoding for the target column
- Train/Test Split
- One Hot Encoding for categorical features
- Feature Scaling for numeric features
- Pipeline implementation with `ColumnTransformer` and `Pipeline`

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Load the Cleaned Dataset

In [ ]:
dataset_path = "electronic_online_shop_output/electronic_online_shop_cleaned.csv"

df = pd.read_csv(dataset_path)

print("Dataset shape:", df.shape)
df.head()

## 3. Dataset Overview

Before preprocessing, check columns, missing values, duplicates, target classes, and data types.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

print("\nDemandLevel distribution:")
print(df["DemandLevel"].value_counts())

## 4. Feature and Target Separation

`DemandLevel` is the target column. `ProductID` and `ProductName` are removed from the feature set because they are identifiers/high-cardinality text fields and usually do not help a basic ML model.

In [ ]:
target_column = "DemandLevel"
columns_to_drop = ["ProductID", "ProductName", target_column]

X = df.drop(columns=columns_to_drop)
y = df[target_column]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)
X.head()

## 5. Label Encoding

The target variable `DemandLevel` contains text labels: `Low`, `Medium`, and `High`. Machine Learning algorithms need numeric target values, so we use `LabelEncoder`.

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

label_mapping = {
    label: int(encoded_value)
    for label, encoded_value in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))
}

print("Label Encoding Mapping:")
print(label_mapping)

pd.DataFrame({"DemandLevel": y.head(10), "EncodedDemandLevel": y_encoded[:10]})

## 6. Identify Numeric and Categorical Features

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 7. Train/Test Split

The dataset is split into training and testing sets. Stratification keeps the class distribution similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\nTrain target distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nTest target distribution:")
print(pd.Series(y_test).value_counts().sort_index())

## 8. One Hot Encoding and Feature Scaling

- `OneHotEncoder` converts categorical features such as `Category` and `Brand` into numeric binary columns.
- `StandardScaler` scales numeric columns so they have mean 0 and standard deviation 1.
- `ColumnTransformer` applies the correct preprocessing to each feature type.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

## 9. Apply the Preprocessing Pipeline

Fit the preprocessing pipeline only on the training set, then transform both train and test sets. This avoids data leakage.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

## 10. View Processed Feature Names

After one hot encoding, categorical columns become multiple binary columns.

In [ ]:
encoded_categorical_names = preprocessor.named_transformers_["categorical"] \
    .named_steps["onehot"] \
    .get_feature_names_out(categorical_features)

processed_feature_names = numeric_features + encoded_categorical_names.tolist()

processed_train_df = pd.DataFrame(X_train_processed, columns=processed_feature_names)

print("Total processed features:", len(processed_feature_names))
processed_train_df.head()

## 11. Full Machine Learning Pipeline

This pipeline combines preprocessing and a classifier in one object. The model here is `LogisticRegression`, used as a clean baseline classifier.

In [ ]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

model_pipeline.fit(X_train, y_train)

print("Full ML pipeline trained successfully.")

## 12. Quick Baseline Evaluation

This is not the final modeling phase, but it confirms that the preprocessing pipeline works correctly from raw features to model prediction.

In [ ]:
y_pred = model_pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Baseline Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 13. Final Preprocessing Summary

- `DemandLevel` was label encoded.
- Categorical features were one hot encoded.
- Numeric features were scaled using `StandardScaler`.
- Train/test split used 80% training and 20% testing.
- Stratification preserved the target class distribution.
- A complete `Pipeline` was implemented to prevent data leakage and keep preprocessing reproducible.